# Part 4 · Variational Algorithms & Optimization Models

Two real deliverables by the end of this notebook:

1. **You will compute the ground-state energy of the H$_2$ molecule** (hydrogen, the simplest molecule there is) **to chemical accuracy.** This exact computation is quantum chemistry's hello world: in 2016 a Google/UCSB team ran it on a real superconducting chip.
2. **You will place four microservices across two servers with a quantum circuit**, by declaring the problem the way you would for any solver (variables, objective, constraint) and letting QiliSDK compile it into physics.

Parts 2 and 3 gave you two ways to *drive* a quantum computer: discrete gates and continuous evolution. This part adds the third ingredient of the near-term playbook: **training**. Instead of writing the perfect circuit by hand, you write a circuit with **tunable knobs** and let a classical optimizer turn them, exactly like fitting a model's weights.

On the architecture map (`overall.jpg`) we light up the last primitives: **Cost Function** and **Optimizer**, composed with the circuits, Hamiltonians, and readouts you already know into the **`VariationalProgram`** workflow, plus the core **`Model`** layer that turns business problems into Hamiltonians.

By the end you will be able to:

- explain the variational loop as an ML training loop with a quantum model inside;
- run **VQE** (Variational Quantum Eigensolver) on a real molecular Hamiltonian and judge the result against chemical accuracy;
- declare an optimization problem with **`Model`**, compile it via **QUBO** into an Ising Hamiltonian, and avoid the two classic decoding traps (penalty weight, variable order);
- solve it with **QAOA** and cross-check the answer with brute force.

In [ ]:
# ▶ Run me first. No-op if QiliSDK is already installed; installs it on a fresh env (e.g. Google Colab).
try:
    import qilisdk
except ImportError:
    import sys
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "qilisdk[openqasm,qir]==0.2.1", "matplotlib", "numpy"], check=True)
    import qilisdk  # Colab: if this still fails, Runtime > Restart session, then rerun
print("QiliSDK", qilisdk.__version__)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 4.0 · The variational idea

Two vocabulary glosses first, because they explain *why* this whole family of algorithms exists:

- **NISQ** (Noisy Intermediate-Scale Quantum): the current hardware era. Tens to a few hundred qubits, no error correction, every gate slightly imperfect.
- **Decoherence**: a qubit does not hold its quantum state forever; interaction with the environment gradually erases it, like a leaky capacitor losing charge. It puts a hard ceiling on how *long* (how deep) a circuit can run before the output turns to mush. Part 5 measures this properly.

Long, exact algorithms (the textbook famous ones) need far more clean gates than NISQ devices can deliver. **Variational quantum algorithms** are the workaround: keep the quantum circuit *short*, make it *tunable*, and put a classical optimizer in charge. The loop:

1. Choose a **parameterized circuit**, called the **ansatz** (German for "starting approach"): a fixed gate layout $U(\vec\theta)$ whose rotation angles $\vec\theta$ are free knobs. It prepares the state $|\psi(\vec\theta)\rangle = U(\vec\theta)\,|0\cdots0\rangle$.
2. Define a **cost** to minimize. Usually the average value of an observable $H$ in that state, $C(\vec\theta) = \langle\psi(\vec\theta)|\,H\,|\psi(\vec\theta)\rangle$. Reminder from Part 1: that bracket is just *row vector @ matrix @ column vector = one real number*.
3. Evaluate $C(\vec\theta)$ **on the quantum device**, let a **classical optimizer** propose new angles, repeat until the cost stops improving.

If you have ever trained a model, you already know this loop. The mapping is one to one:

| variational algorithm | ML training loop |
|---|---|
| ansatz $U(\vec\theta)$ | model architecture with trainable weights |
| cost $\langle H\rangle$ | loss function |
| `SciPyOptimizer` | the optimizer (it literally wraps `scipy.optimize.minimize`) |
| convergence curve | training loss curve |
| backend (`QiliSim`, a QPU) | the device you train on |

The division of labor is the point: the quantum processor evaluates the cost of a state living in a $2^n$-dimensional space (the part classical computers choke on), while the classical loop does the bookkeeping and the search (the part classical computers are great at). QiliSDK bundles the three pieces into one **`VariationalProgram(functional, optimizer, cost_function)`**, and you run it with the exact same `backend.execute(functional, readout)` call as everything else in this tutorial.

## 4.1 · VQE: the Variational Quantum Eigensolver

**VQE** hunts the *lowest* eigenvalue of an observable $H$: the **ground-state energy** (Part 3 vocabulary: an eigenstate is a state whose measurement outcome is certain, its eigenvalue is that outcome, and the ground state is the eigenstate with the smallest energy). VQE rests on one inequality, the **variational principle**:

$$\langle\psi(\vec\theta)|\,H\,|\psi(\vec\theta)\rangle \;\ge\; E_{\text{ground}} \quad\text{for every state } |\psi(\vec\theta)\rangle.$$

Why it is true, in one sentence: $\langle H\rangle$ is a probability-weighted **average** of $H$'s eigenvalues, and no average can undercut the smallest value in the mix. The consequence is a safety net most numerical methods do not have: whatever circuit you try, honest or terrible, its energy can only sit *above* the true answer. Minimizing $\langle H\rangle$ therefore walks the circuit down toward the ground state, and every step of progress is a *certified improvement of an upper bound*.

We warm up on a toy cost before the real molecule. Three ingredients:

- **Ansatz**: `HardwareEfficientAnsatz` stacks, per layer, one generic rotation on every qubit plus entangling gates between neighbors. The rotation is **`U3`**, the fully general single-qubit gate with three angles (any single-qubit gate whatsoever equals some `U3(theta, phi, gamma)`). Its parameters are created for you with auto-generated names.
- **Cost function**: `ObservableCostFunction(H)` computes $\langle H\rangle$. The observable is built from the **analog** Pauli operators of Part 3, and since this notebook mixes them with digital gate classes of the same names (`Z` the operator vs `Z` the gate), we alias the analog imports: `from qilisdk.analog import Z as aZ`.
- **Optimizer**: `SciPyOptimizer(method="COBYLA")`. COBYLA is a gradient-free method from `scipy.optimize`, a solid default for a handful of parameters.

The warm-up cost is $\langle Z_0 Z_1\rangle$. Part 3 taught you to read this: $Z_iZ_j$ scores $+1$ when two bits agree and $-1$ when they differ, so the floor is $-1$, reached by any state where the qubits always disagree. First, look inside the ansatz black box:

In [ ]:
from qilisdk.analog import Z as aZ                # analog Pauli operator (aliased: digital Z is a gate class)
from qilisdk.digital import CNOT, U3, HardwareEfficientAnsatz
from qilisdk.functionals import DigitalPropagation, VariationalProgram
from qilisdk.optimizers import SciPyOptimizer
from qilisdk.cost_functions import ObservableCostFunction
from qilisdk.backends import QiliSim
from qilisdk.readout import Readout

ansatz = HardwareEfficientAnsatz(nqubits=2, layers=1, one_qubit_gate=U3, two_qubit_gate=CNOT)
print("trainable parameters:", ansatz.get_parameter_names())
ansatz.draw()

Inside, the ansatz is straightforward: one `U3` per qubit (three named angles each, six parameters total) and one `CNOT` to entangle them. The parameter names (`U3(0)_theta_0`, ...) matter: they are the keys you use to set angles by hand, and the order they are listed in is the order the optimizer reports its results in.

Now wire the three pieces into a `VariationalProgram` and run it. We read the cost with **state tomography** (from Part 2: on a simulator we can take the exact state, so the numbers below are deterministic, with no shot noise). `store_intermediate_results=True` records the cost at every optimizer step, which gives us the training loss curve.

In [ ]:
program = VariationalProgram(
    functional=DigitalPropagation(ansatz),                 # what to run: the tunable circuit
    optimizer=SciPyOptimizer(method="COBYLA", tol=1e-6),   # the classical search
    cost_function=ObservableCostFunction(aZ(0) * aZ(1)),   # the loss: <Z0 Z1>, floor at -1
    store_intermediate_results=True,                       # keep the loss curve
)
result = QiliSim().execute(program, Readout().with_state_tomography())
print("optimal cost       :", round(result.optimal_cost, 6))
print("circuit evaluations:", len(result.intermediate_results))

The optimizer reached the floor at $-1$ in a few dozen circuit evaluations (each evaluation is one full run: prepare the state with the current angles, then read the cost). Plotting the recorded costs gives a **training loss curve**, the same diagnostic you would use for any machine-learning model:

In [ ]:
costs = [step.cost for step in result.intermediate_results]
plt.plot(costs, marker=".")
plt.xlabel("circuit evaluation")
plt.ylabel(r"cost  $\langle Z_0 Z_1 \rangle$")
plt.axhline(-1.0, ls="--", c="gray", label="true minimum")
plt.title("VQE convergence: a quantum training loss curve")
plt.legend()
plt.show()

### The main application: the energy of a real molecule

We now point this machinery at a real problem. A molecule's **ground-state energy** is a central quantity in chemistry: differences between such energies determine reaction rates, drug binding strengths, battery voltages, and catalyst efficiencies. Computing them for large molecules is the application Feynman had in mind when he proposed quantum computers (Part 1), because the electrons in a molecule are themselves a quantum system that classical computers can only approximate.

In 2016, a Google/UCSB team ran exactly the computation below on a real superconducting chip: the **H$_2$ molecule** (two hydrogen atoms), reduced by chemists to a **2-qubit Hamiltonian** with six published coefficients (O'Malley et al., *PRX* **6**, 031007 (2016), bond length 0.7414 angstrom). We take those coefficients as given: mapping electrons to qubit operators is its own subfield, and the purpose of the published numbers is precisely that you can start from them. Two units to know:

- **Hartree (Ha)**: the atomic unit of energy in which chemists quote molecular energies (1 Ha is about 27.2 eV).
- **Chemical accuracy**: 1.6 mHa (milli-Hartree). An energy within 1.6 mHa of the true value is accurate enough for chemically meaningful predictions; a larger error can throw off the reaction-rate estimates built on it by orders of magnitude.

The plan: build $H_{\text{mol}}$ from the analog Paulis, get the exact answer by brute-force diagonalization (possible only because this 2-qubit toy has a $4\times4$ matrix), then let VQE find the same answer the scalable way, with a 2-layer ansatz.

*(⏩ This cell runs a full training loop, roughly 800 circuit evaluations, and takes a few seconds. If you are reading along without running it, the solutions notebook includes the output.)*

In [ ]:
from qilisdk.analog import I as aI, X as aX, Y as aY, Z as aZ   # analog Paulis, aliased
from qilisdk.digital import CNOT, U3, HardwareEfficientAnsatz
from qilisdk.functionals import DigitalPropagation, VariationalProgram
from qilisdk.optimizers import SciPyOptimizer
from qilisdk.cost_functions import ObservableCostFunction
from qilisdk.backends import QiliSim
from qilisdk.readout import Readout

# H2 at bond length 0.7414 angstrom, reduced to 2 qubits (O'Malley et al., PRX 6, 031007 (2016))
g0, g1, g2, g3, g4, g5 = -0.4804, 0.3435, -0.4347, 0.5716, 0.0910, 0.0910
H_mol = (g0 * aI(0) + g1 * aZ(0) + g2 * aZ(1) + g3 * aZ(0) * aZ(1)
         + g4 * aY(0) * aY(1) + g5 * aX(0) * aX(1))

# The reference answer, by brute force: only possible because 2 qubits = a 4x4 matrix
exact = np.linalg.eigvalsh(H_mol.to_matrix().toarray()).min()

ansatz = HardwareEfficientAnsatz(nqubits=2, layers=2, one_qubit_gate=U3, two_qubit_gate=CNOT)
program = VariationalProgram(
    functional=DigitalPropagation(ansatz),
    optimizer=SciPyOptimizer(method="COBYLA", tol=1e-9),
    cost_function=ObservableCostFunction(H_mol),
    store_intermediate_results=True,
)
result = QiliSim().execute(program, Readout().with_state_tomography())

print(f"exact ground energy : {exact:.6f} Ha")
print(f"VQE energy          : {result.optimal_cost:.6f} Ha")
print(f"VQE error           : {abs(result.optimal_cost - exact):.2e} Ha   (chemical accuracy: 1.6e-03 Ha)")
print(f"circuit evaluations : {len(result.intermediate_results)}")

Read the two energy lines: they agree at $-1.851199$ Ha to all printed decimals. Across our test runs the VQE error landed between $10^{-12}$ and $10^{-9}$ Ha, roughly *a million times tighter* than the 1.6 mHa chemical-accuracy bar. A 12-parameter circuit, trained by plain COBYLA, reproduced a real molecule's electronic ground-state energy.

One chemistry footnote for completeness: $H_{\text{mol}}$ describes the **electrons**. The two nuclei also repel each other with a fixed $+0.7137$ Ha at this geometry, so the molecule's total energy is $-1.851199 + 0.7137 \approx -1.1375$ Ha, matching the textbook value for this standard calculation.

This is where the value of VQE becomes clear. Look at which line of the code stops working as molecules grow:

```python
exact = np.linalg.eigvalsh(H_mol.to_matrix().toarray()).min()
```

That line builds the full $2^n \times 2^n$ matrix. At 2 qubits it is $4\times4$; at 50 qubits it would have about $10^{30}$ entries, more numbers than any datacenter can store. So the exact-diagonalization line is the part that fails to scale, while the VQE loop is the part that keeps working: it never builds the full matrix, it only *prepares states and measures their energy*, which is exactly what a quantum device does natively. At this toy size, brute force is still far faster. The point is that you just ran the real workflow, the one that transfers unchanged to molecules where brute-force diagonalization is impossible, once hardware is good enough to run deep circuits cleanly (Part 5 quantifies how good).

### 🧩 Exercise 4.1: see the variational principle hold

**Goal:** verify the variational principle directly. Bind *random* angles into the ansatz three times, measure each random circuit's energy, and confirm that every one sits **above** the exact ground energy.

**Why it matters:** the inequality $\langle H\rangle \ge E_{\text{ground}}$ is what makes VQE trustworthy. It means that a poor ansatz, a weak optimizer, or bad luck can only push the answer *too high*, never below the true value, so the optimizer's task is well defined: descend toward a floor it cannot cross. Testing the inequality on circuits you randomized yourself makes that guarantee concrete.

**Your task** (`H_mol` and `exact` are still in scope from the molecule cell above):

1. Build a fresh `HardwareEfficientAnsatz` (2 qubits, 2 layers, `U3` + `CNOT`) and read its parameter names with `get_parameter_names()`.
2. Three times over: draw one random angle per parameter (`np.random.default_rng()` and `rng.uniform(0, 2 * np.pi, size=...)` do nicely) and bind them with `set_parameters`, which takes a dict mapping each parameter name to a value.
3. After each bind, evaluate the circuit's exact energy: execute it like any fixed circuit from Part 2, with an **expectation readout** on `H_mol` (`nshots=0` means exact, and `get_expectation_values()` returns one value per observable).
4. Print each random energy next to `exact` and check that the inequality held all three times.

*Hint: no `VariationalProgram` is needed here, since there is nothing to train. In this exercise you are the (deliberately bad) optimizer.*

In [ ]:
rng = np.random.default_rng()   # reroll as often as you like

# H_mol, exact, and all imports are in scope from the flagship cell above.

# TODO 1: build a fresh HardwareEfficientAnsatz (2 qubits, 2 layers, U3 + CNOT)
#         and read its parameter names

# TODO 2: three times: draw one random angle in [0, 2*pi) per parameter and bind
#         them with set_parameters (a dict of name -> value)

# TODO 3: after each bind, evaluate the exact energy of the circuit
#         (expectation readout on H_mol with nshots=0)

# TODO 4: print each random energy next to `exact`: every one should sit ABOVE it

## 4.2 · The Model layer: state the problem, compile the physics

In Part 3 you encoded a scheduling problem into a Hamiltonian **by hand**: one $Z_iZ_j$ term per conflict, all weights equal, no constraints. Hand-encoding stops scaling the moment a real problem shows up with *weights* ("this conflict matters more than that one") and *constraints* ("exactly two per group"). QiliSDK's **`Model`** layer is the compiler for that job: you declare variables, an objective, and constraints, the way you would for any classical solver, and it lowers the whole thing to a Hamiltonian.

A problem every backend developer has met. Four microservices must be split across **two servers**. Services that talk a lot should sit on the *same* server (cross-server calls pay network latency and egress). For load balance, each server must host **exactly two** services. The traffic matrix (requests per second between service pairs):

| pair | traffic |
|---|---|
| auth ↔ api | 120 |
| api ↔ db | 300 |
| api ↔ cache | 200 |
| db ↔ cache | 250 |
| auth ↔ db | 20 |

**Encoding the objective.** One binary variable per service: $x_s = 0$ means server A, $x_s = 1$ means server B. For a pair $(a, b)$, the polynomial

$$x_a + x_b - 2\,x_a x_b$$

equals $1$ exactly when the two ends **differ** and $0$ when they match (truth-table it: $0{+}0{-}0 = 0$, $\;1{+}1{-}2 = 0$, $\;1{+}0{-}0 = 1$). Graph theorists call this the **cut** indicator, the heart of the classic Max-Cut problem; here it is simply "does this pair's traffic cross between servers". Multiply each indicator by its traffic and sum: that is the total cross-server traffic, our objective to **minimize**.

**Encoding the constraint.** "Exactly two services per server" is just $\sum_s x_s = 2$. We attach a `lagrange_multiplier` of 1000 to it, a *penalty weight* whose size we will justify in a moment (spoiler: the default would silently give a wrong answer).

In [ ]:
from qilisdk.core import Model, BinaryVariable, ObjectiveSense
from qilisdk.core.variables import EQ

services = ["auth", "api", "db", "cache"]
traffic = {("auth", "api"): 120, ("api", "db"): 300, ("api", "cache"): 200,
           ("db", "cache"): 250, ("auth", "db"): 20}

x = {s: BinaryVariable(f"x_{s}") for s in services}      # x_s = 0 -> server A, x_s = 1 -> server B

# total cross-server traffic: each pair's traffic counts exactly when its endpoints differ
cross = sum(w * (x[a] + x[b] - 2 * x[a] * x[b]) for (a, b), w in traffic.items())

m = Model("placement")
m.set_objective(cross, sense=ObjectiveSense.MINIMIZE)
m.add_constraint("balance", EQ(sum(x[s] for s in services), 2), lagrange_multiplier=1000)
print(m)

### From `Model` to QUBO to Hamiltonian

Quantum optimizers (annealers, QAOA) want the problem in one specific shape: a **QUBO**, *Quadratic Unconstrained Binary Optimization*. In Python terms: minimize a polynomial in bits where every term is at most a product of two, something like `sum(Q[i][j] * x[i] * x[j] for i, j in ...)` (the linear-algebra crowd writes it $x^\top Q\, x$). Note the U: **unconstrained**. `model.to_qubo()` does the lowering:

- **Constraints become penalty terms.** Our equality turns into $1000\,(\sum_s x_s - 2)^2$, added to the objective: zero when the constraint holds, painfully positive when it does not. (An *inequality* constraint would additionally mint **slack variables**, extra helper bits, so in general expect the compiled problem to use more qubits than you have decision variables. Our equality needs none.)
- **Higher-degree terms get linearized.** Any product of three or more bits is rewritten quadratically via a standard trick (Rosenberg penalties); the takeaway is simply that it, too, costs extra qubits. Our objective is already quadratic, so this pass is a no-op here.

Then `qubo.to_hamiltonian()` swaps each bit for a spin with the substitution $x_i = (1 - Z_i)/2$, which maps $x_i = 0 \leftrightarrow Z_i = +1$ and $x_i = 1 \leftrightarrow Z_i = -1$: the same $Z$ eigenvalue convention you used all through Part 3. Out comes an **Ising Hamiltonian whose ground state is the optimal placement**.

> ⚠️ **Decoding trap, memorize this one.** Qubit $i$ of that Hamiltonian corresponds to entry $i$ of `qubo.qubo_objective.variables()`, the *compiled* variable order, and in this build that order comes out **alphabetical by label**: `x_api` is qubit 0, *not* `x_auth`, even though we declared `auth` first. Decode by declaration order and every service lands on the wrong server with no error raised. Always read the order off the compiled QUBO, as the next cell does.

In [ ]:
qubo = m.to_qubo()                # constraint -> penalty term, everything quadratic in bits
ham = qubo.to_hamiltonian()       # bits -> spins: an Ising Hamiltonian, Part 3 style

var_order = [v.label for v in qubo.qubo_objective.variables()]   # qubit i <-> var_order[i]
print("qubits     :", ham.nqubits)
print("qubit order:", var_order)
print(ham)

The printed Hamiltonian is a constant plus pure $Z_iZ_j$ couplings, exactly the shape you hand-built in Part 3, except that a compiler wrote it, with the weights and penalty already folded in.

### Read the answer straight off the diagonal

Because the compiled Hamiltonian contains only $Z$s (and a constant), Part 3's key fact applies: **in the computational basis it is a diagonal matrix**, a literal lookup table from bitstring to cost. Every classical assignment is an eigenstate, and its diagonal entry is its objective value plus any penalty. So at 4 qubits ($2^4 = 16$ entries) we can find the optimum with `np.argmin`, and, because the compiler kept the constant offset, each entry is in the problem's *own units*: the ground value below is an actual number of requests per second of cross-server traffic.

Reading the diagonal is brute force, just expressed in the language of the Hamiltonian, and at 4 qubits brute force is still the fastest option. The reason to do it is the habit from Part 3: **certify the encoding while it is still cheap**, before handing the Hamiltonian to any quantum solver. If the encoding is wrong, a perfect solver returns a perfect answer to the wrong question. We also cross-check against a plain-Python enumeration of all balanced splits, so no step here rests on trust:

In [ ]:
import itertools

diag = np.diag(ham.to_matrix().toarray()).real       # Z-only Hamiltonian -> its matrix is diagonal
k = int(np.argmin(diag))
best_bits = format(k, f"0{ham.nqubits}b")            # big-endian: character i of the string = qubit i

assignment = {label.removeprefix("x_"): ("A" if b == "0" else "B")
              for label, b in zip(var_order, best_bits)}     # decode via the COMPILED order

def cross_traffic(assign):
    return sum(w for (a, b), w in traffic.items() if assign[a] != assign[b])

print("ground-state bitstring:", best_bits, "  energy:", diag[k])
print("placement             :", assignment)
print("cross traffic         :", cross_traffic(assignment), "req/s")
print()

# cross-check: enumerate every balanced split classically
print("all balanced splits (pair listed = server A):")
for pair in itertools.combinations(services, 2):
    trial = {s: ("A" if s in pair else "B") for s in services}
    print(f"  {' + '.join(pair):13s} | cross traffic {cross_traffic(trial)}")

The ground state says: **auth and api on one server, db and cache on the other, 520 req/s crossing**, and the enumeration confirms that no balanced split does better. (Two rows show 520 because putting `auth + api` on server A and putting `db + cache` on server A describe the *same* split with the server labels swapped.)

### The penalty weight: a production lesson

We set `lagrange_multiplier=1000`. The default is 100. Consider what the default would have done; this is arithmetic, with no physics needed:

- **What breaking the constraint saves:** ignore balance and put `auth` alone on a server. Only auth's own flows (120 + 20 = 140) then cross, versus 520 for the best balanced split. Saving: **380**.
- **What breaking the constraint costs:** a 3-to-1 split misses $\sum x_s = 2$ by one, so the penalty term $100\,(\sum x_s - 2)^2$ charges $100 \cdot 1^2 =$ **100**.

The saving (380) exceeds the penalty (100), so with the default weight the *true* ground state of the compiled Hamiltonian is the **unbalanced** split (energy $140 + 100 = 240$, below 520). Nothing raises an error. The annealer or QAOA would faithfully find that ground state, you would decode a placement that violates the load-balance requirement, and you would only discover the violated constraint once the system was running in production. The next cell demonstrates this by compiling the weak-penalty model and reading its diagonal.

This is a genuine tuning parameter in industrial QUBO pipelines, not a toy detail. The penalty must exceed anything a constraint violation could ever save (here 1000 > 520 is comfortably enough), but it should not be orders of magnitude larger than necessary, or the objective's real differences shrink to rounding errors next to the penalty scale and the solver's landscape flattens out.

In [ ]:
# Same model, but with the DEFAULT penalty weight (100). Fresh variables for a fresh model.
x2 = {s: BinaryVariable(f"x_{s}") for s in services}
cross2 = sum(w * (x2[a] + x2[b] - 2 * x2[a] * x2[b]) for (a, b), w in traffic.items())

weak = Model("placement_weak")
weak.set_objective(cross2, sense=ObjectiveSense.MINIMIZE)
weak.add_constraint("balance", EQ(sum(x2[s] for s in services), 2))   # default lagrange_multiplier = 100

weak_qubo = weak.to_qubo()
weak_diag = np.diag(weak_qubo.to_hamiltonian().to_matrix().toarray()).real
k2 = int(np.argmin(weak_diag))
weak_bits = format(k2, "04b")
weak_order = [v.label for v in weak_qubo.qubo_objective.variables()]

print("weak-penalty ground state:", weak_bits, "  energy:", weak_diag[k2])
print("decoded:", {label.removeprefix("x_"): ("A" if b == "0" else "B")
                   for label, b in zip(weak_order, weak_bits)})
print("-> an unbalanced 3-to-1 split: the constraint got bought off")

## 4.3 · QAOA: solve it on a circuit

Reading the diagonal stops being feasible at scale for the same reason `eigvalsh` did: the $2^n$ entries. So we turn to the quantum solver. **QAOA** (Quantum Approximate Optimization Algorithm) is the variational algorithm designed for problem Hamiltonians like ours. Its circuit alternates two kinds of layers, each with a clear role:

1. **Start in uniform superposition** ($H$ on every qubit): all 16 placements present at once, with equal amplitude.
2. **Problem layer** $e^{-i\gamma H_p}$: recall from Part 3 that $e^{-iHt}$ is the matrix exponential, i.e. evolution under $H$ for time $t$. Because our $H_p$ is diagonal, this layer cannot move amplitude between bitstrings; it can only rotate each bitstring's **phase**, by an angle proportional to *that bitstring's cost*. After one layer, every one of the 16 candidates has its cost encoded in its phase.
3. **Mixer layer** $e^{-i\alpha H_m}$ with $H_m = \sum_i X_i$: $X$ flips bits, so this layer moves amplitude between neighboring bitstrings. Moving amplitude between components whose phases differ is what produces **interference**, and interference converts the invisible phase differences into visible probability differences.
4. Repeat the pair $p$ times. The $2p$ angles $\gamma_1..\gamma_p,\ \alpha_1..\alpha_p$ are the trainable parameters; minimizing $\langle H_p\rangle$ over them tunes the interference so that expensive placements cancel and cheap ones accumulate.

This is the concrete payoff of Part 1's measurement rule: measurement will only ever return 4 classical bits, so the circuit must arrange interference to make the *right* 4 bits the likely ones before we measure. (If you took Part 3's optional Trotterization detour: a QAOA circuit is exactly a Trotterized anneal whose step sizes have been unfrozen and handed to an optimizer.)

**One practical step before running, borrowed directly from machine learning: rescale the cost.** Our Hamiltonian's coefficients sit at the penalty scale, $O(1000)$, while COBYLA proposes angle steps of order 1; on that mismatched landscape our unscaled test runs converged to a *wrong* split more often than not. Multiplying a Hamiltonian by a positive constant rescales every energy by that constant but leaves the ordering and the ground state unchanged, just as dividing a loss by 1000 changes no argmin. After scaling by $1/1000$, every test run we made converged to the correct placement.

*(⏩ This training loop takes a few seconds. Caveat on reproducibility: optimizer runs can differ slightly from one run to the next; if your top bitstring ever fails the balance check below, rerun the cell. With the rescaled Hamiltonian we did not see that happen in testing.)*

In [ ]:
from qilisdk.digital import QAOA

ham_n = (1.0 / 1000.0) * ham          # rescale: same ground state, optimizer-friendly numbers

qaoa = QAOA(ham_n, layers=3)          # p = 3 -> six trainable angles
print("QAOA parameters:", qaoa.get_parameter_names())
print("circuit size   :", len(qaoa.gates), "gates")

program = VariationalProgram(
    functional=DigitalPropagation(qaoa),
    optimizer=SciPyOptimizer(method="COBYLA", tol=1e-6),
    cost_function=ObservableCostFunction(ham_n),
)
result = QiliSim().execute(program, Readout().with_state_tomography())

# the full output distribution of the trained circuit (exact, since we use tomography)
probs = result.optimal_execution_results.state_tomography.probabilities
print("\ntop bitstrings after training:")
for bits, p in sorted(probs.items(), key=lambda kv: -kv[1])[:4]:
    print(f"  {bits}: {p:.3f}")

Interpret the output distribution. The two front-runners, at about $0.17$ each, are **mirror images** of each other: flipping every bit swaps the labels A and B, and nothing in the cost depends on which server is called which, so the two optimal bitstrings are degenerate (they have equal energy). Just behind them sit *other balanced splits* (in our runs, `0110`/`1001`, the 590 req/s split, at about $0.16$), while the penalty term has pushed the unbalanced strings far down the list. So QAOA at $p = 3$ *concentrates* probability on good answers, with the optimum ahead by a small margin; it does not sharpen to certainty. That is acceptable in practice, because checking a candidate is classically cheap: you score each sampled bitstring with the objective and keep the best one you saw, so even a modest tilt toward the optimum is enough. On a simulator we read the exact distribution via tomography; on real hardware you would run `with_sampling(nshots=...)` instead, and the training cost would then carry shot noise, making it an estimate rather than an exact average.

Decode the winner with `var_order` (the compiled order from section 4.2, never the declaration order) and score it with the classical `cross_traffic` checker:

In [ ]:
winner = max(probs, key=probs.get)
assignment = {label.removeprefix("x_"): ("A" if b == "0" else "B")
              for label, b in zip(var_order, winner)}
server_a = sorted(s for s, srv in assignment.items() if srv == "A")
server_b = sorted(s for s, srv in assignment.items() if srv == "B")

print("winner bitstring:", winner)
print("server A        :", server_a)
print("server B        :", server_b)
print("cross traffic   :", cross_traffic(assignment), "req/s   (diagonal read said 520)")
print("balanced        :", len(server_a) == len(server_b))

### 🧩 Exercise 4.2: rewire the traffic, predict the split

**Goal:** the traffic matrix changes; you predict the new optimal placement *on paper first*, then make the full pipeline (Model → QUBO → Hamiltonian → solve → decode) confirm it.

**The change:** a refactor rendered the db ↔ cache chatter obsolete (250 → **0**), and a new audit feature makes auth hammer db (20 → **400**):

**Why it matters:** a solver you cannot sanity-check at small size is a solver you cannot trust at large size. This is also the test of whether the encoding really lives in your head: if you can reason "these flows pull those services together", you own the objective, not just the API.

**Your task:**

1. *On paper:* with flows auth↔api 120, api↔db 300, api↔cache 200, auth↔db 400, which pairing should share servers? Tally what each candidate split lets *not* cross. Write your prediction down before running anything.
2. Rebuild the `Model` with `new_traffic` (fresh `BinaryVariable`s for a fresh model, same balance constraint, `lagrange_multiplier=1000`).
3. Compile to QUBO and Hamiltonian, and record the qubit order from the compiled QUBO.
4. Solve it: rescale and rerun QAOA as in 4.3 (or read the diagonal as in 4.2, both are fair).
5. Decode the winning bitstring into a placement, print its cross traffic, and confirm against a brute-force pass over `itertools.combinations(services, 2)`.

*Hint for the paper step: db now has two heavy relationships, 400 with auth and 300 with api, but only one service can share db's server. Which one should it be, and who does that force together on the other server?*

In [ ]:
import itertools

new_traffic = {("auth", "api"): 120, ("api", "db"): 300, ("api", "cache"): 200,
               ("db", "cache"): 0, ("auth", "db"): 400}

# Your prediction, BEFORE running anything:
# prediction = {"auth": "?", "api": "?", "db": "?", "cache": "?"}

# TODO 1: rebuild the Model: fresh BinaryVariables, the cut-polynomial objective
#         over new_traffic, and the balance constraint with lagrange_multiplier=1000

# TODO 2: compile to QUBO and Hamiltonian; record the qubit order from
#         qubo_objective.variables()

# TODO 3: solve it (rescale + QAOA as in 4.3, or read the diagonal as in 4.2)

# TODO 4: decode the winning bitstring into a placement and print its cross traffic

# TODO 5: brute-force all balanced splits with itertools.combinations to confirm

## Recap & what's next

- **The variational loop is a machine-learning training loop with a quantum model inside**: ansatz = architecture with weights, $\langle H\rangle$ = loss, `SciPyOptimizer` = the optimizer (wrapping `scipy.optimize.minimize`), convergence curve = loss curve. QiliSDK packages it as `VariationalProgram(functional, optimizer, cost_function)`, run by the same `backend.execute(...)` as everything else.
- **VQE** used the variational principle ($\langle H\rangle \ge E_{\text{ground}}$, since an average cannot fall below its smallest ingredient) to compute the H$_2$ molecule's ground-state energy at $-1.851199$ Ha, about a million times inside chemical accuracy; your Exercise 4.1 circuits confirmed the inequality directly. The `eigvalsh` reference line is the part that fails to scale (its cost grows as $2^n$); the measure-and-descend loop is the part that scales.
- **The `Model` layer** compiled a microservice-placement problem (variables, weighted objective, balance constraint) through **QUBO** into an Ising Hamiltonian, and demonstrated two production lessons: the **penalty weight** must exceed anything a constraint violation can save (the default 100 quietly returned an unbalanced split; 1000 fixed it), and decoding must follow the **compiled variable order**, not the declaration order.
- **QAOA** solved it on a circuit: cost encoded as phase, the mixer turning phase into interference, six trained angles concentrating about a third of the probability on the two mirror-image optimal placements, with brute force agreeing at 520 req/s (and at 420 after your Exercise 4.2 rewiring). Plus one standard trick: rescale the Hamiltonian just as you would rescale a loss.
- At 2 to 4 qubits, `eigvalsh` and `itertools` are still faster than any quantum solver here. What you built is the workflow that transfers unchanged to sizes where they cannot run.

**Part 5, Noise & Realistic Execution:** everything so far ran on a mathematically perfect simulator. Real chips flip bits, lose phase, and decay mid-circuit. Part 5 models those processes and answers a question this part set up: **how much gate noise can the H$_2$ result tolerate before it stops clearing chemical accuracy?** That number is a noise budget for a real application, and it is why hardware teams pursue every additional decimal of gate fidelity.